# Open Notebook & Additional Resources

<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ORM_AI_Agents_Bootcamp/blob/main/hands_on/DAY_2_HANDS_ON_SESSION_1_stakeholder_chaos_to_structure.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
<a target="_blank" href="https://learning.oreilly.com/library/view/ai-agents-the/0642572247775/">
  <img src="https://img.shields.io/badge/AI%20Agents%20Book-Read%20on%20O'Reilly-d40101?style=flat" alt="AI Agents Book – Read on O'Reilly"/>
</a>





#<font color="red" size="10">
<b>HANDS-ON TIME: 15 mins</b>
</font>

# About this Notebook

1. **Downstream tools need roles for each fact.** Jira wants components, labels, acceptance criteria; codegen wants inputs/outputs and steps; workflow builders need triggers, escalations, and tool boundaries. `details: list[str]` does not tell you which string is a constraint vs. a feature vs. an open question.

2. **Everything is still interpretation.** The next hop must re-read unstructured text inside `details`, so you have not reduced ambiguity or enabled safe automation.

3. **Minimal upgrade path:** add a discriminant (`task_type`), separate **must** vs **nice-to-have** vs **exclusions**, record **output format** and **human handoff**, and reserve **open_questions** for explicit abstention instead of silent guessing.

In [1]:
!pip install -q "langchain-openai" "python-dotenv>=1.0,<2.0" "pydantic>=2,<3"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 9.6 MB/s eta 0:00:00


In [3]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

NEBIUS_API_KEY = os.environ.get("NEBIUS_API_KEY")

if not NEBIUS_API_KEY:
    print("⚠️ NEBIUS_API_KEY not found.")
    NEBIUS_API_KEY = input("Enter your NEBIUS_API_KEY: ").strip()

NEBIUS_API_KEY = NEBIUS_API_KEY.strip().strip('"').strip("'")

llm = ChatOpenAI(
    model="openai/gpt-oss-120b",
    api_key=NEBIUS_API_KEY,
    base_url="https://api.tokenfactory.nebius.com/v1/",
    temperature=0,
)

resp = llm.invoke([HumanMessage(content="Say hello in one sentence.")])
print(resp.content)

Hello, I hope you’re having a wonderful day!


In [4]:
from __future__ import annotations

import json
import os
import re
from typing import Any, Literal

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field, field_validator


## Reference schema: `TaskSpec`

Compared to the exercise starter, this solution adds:

- **`priority`** — ticketing and sprint tools assume it.
- **`nice_to_have`** — separated from **exclusions** (exclusions = must refuse / out of scope; nice_to_have = hedged or later).
- **`constraints`** — non-functional requirements (keep it simple, performance, compliance).

List values use **snake_case tokens** so labels stay stable across exports (Jira labels, YAML keys, codegen identifiers).

In [5]:
def _snake_token(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    return re.sub(r"_+", "_", s).strip("_") or "item"


class TaskSpec(BaseModel):
    """Contract for turning messy stakeholder text into tool-ready structure.

    Field → downstream use:
    - task_type / priority → triage, boards, routing
    - objective → issue title, README one-liner
    - must_have / nice_to_have → acceptance criteria, milestones
    - exclusions → safety / policy checks, router negatives
    - constraints → NFR checklist, test plan
    - output_format → codegen package layout, export pipeline
    - human_handoff → HITL nodes, escalation rules
    - open_questions → sub-tasks or discovery tickets
    """

    task_type: Literal["dashboard", "chatbot", "script"] = Field(
        description="Primary bucket: analytics UI, conversational assistant, or batch/automation script."
    )
    objective: str = Field(
        description="One sentence outcome in plain language; suitable as an issue title body line."
    )
    priority: Literal["high", "medium", "low"] = Field(
        description="Inferred urgency; use medium when not stated.",
    )
    must_have: list[str] = Field(
        default_factory=list,
        description="Firm requirements or clearly in-scope capabilities (snake_case tokens).",
    )
    nice_to_have: list[str] = Field(
        default_factory=list,
        description="Optional, later, or hedged items (maybe, probably, later) as snake_case tokens.",
    )
    exclusions: list[str] = Field(
        default_factory=list,
        description="Explicit out-of-scope topics or behaviors to refuse / escalate (snake_case).",
    )
    constraints: list[str] = Field(
        default_factory=list,
        description="Non-functional constraints: simple UX, latency, privacy, platforms (snake_case).",
    )
    output_format: str | None = Field(
        default=None,
        description="Stated deliverable medium: pdf_export, cleaned_csv, slack_bot, etc.",
    )
    human_handoff: bool = Field(
        default=False,
        description="True if a human must handle some cases or approvals.",
    )
    open_questions: list[str] = Field(
        default_factory=list,
        description="Unresolved ambiguities; short interrogative phrases, not guesses.",
    )

    @field_validator(
        "must_have",
        "nice_to_have",
        "exclusions",
        "constraints",
        mode="before",
    )
    @classmethod
    def _normalize_tokens(cls, v: Any) -> Any:
        if not isinstance(v, list):
            return v
        return [_snake_token(str(x)) for x in v]

## Configuration + validation

`CONFIG` is the single place facilitators flip during class to show how **policy** (not just prompting) shapes extractions.

In [6]:
CONFIG: dict[str, Any] = {
    "allow_inference": False,
    "collect_open_questions": True,
    "require_must_have": True,
}


def validate_task_spec(spec: TaskSpec) -> list[str]:
    issues: list[str] = []
    if CONFIG.get("require_must_have") and not spec.must_have:
        issues.append("must_have is empty but require_must_have is True.")
    if spec.human_handoff and not spec.exclusions and spec.task_type == "chatbot":
        issues.append(
            "human_handoff is True for a chatbot but exclusions is empty — verify escalation triggers are captured."
        )
    return issues

## System prompt (aligned with `CONFIG`)

The prompt names each schema field so the model maps language to **slots**, not to a generic paragraph.

In [7]:
def build_system_prompt() -> str:
    allow = CONFIG.get("allow_inference", True)
    questions = CONFIG.get("collect_open_questions", True)
    return f"""You extract a TaskSpec from a messy stakeholder message.

Schema slots:
- task_type: dashboard | chatbot | script
- objective: one clear sentence
- priority: high | medium | low (default medium if unstated)
- must_have: non-negotiable capabilities (snake_case list)
- nice_to_have: optional, "later", "maybe", weakly committed items (snake_case)
- exclusions: topics or request types the user said to refuse or not handle (snake_case)
- constraints: non-functional requirements like simplicity, performance, compliance (snake_case)
- output_format: concrete deliverable medium if implied (e.g. pdf_export, cleaned_csv); else null
- human_handoff: true if any scenario must go to a person
- open_questions: what you still need to know; short phrases

Rules:
- Do not invent requirements. If the user did not imply it, it does not belong in must_have.
- Billing/support/legal handoffs usually imply human_handoff true and exclusions should name the topic area.
- Hedged language ("maybe", "probably later") → nice_to_have, not must_have.
- Inference policy: {"you may infer reasonable unstated engineering details" if allow else "do not guess—put unknowns in open_questions"}.
- Open questions policy: {"list every important unresolved ambiguity" if questions else "only include blocking unknowns"}.
"""

## Extraction

In [8]:
def extract_task_spec(message: str) -> TaskSpec:
    chain = llm.with_structured_output(TaskSpec)
    return chain.invoke(
        [SystemMessage(content=build_system_prompt()), HumanMessage(content=message)]
    )

## Downstream formatters

Each function should be **boring**: string formatting only, no second LLM pass.

In [9]:
def task_spec_to_github_issue(spec: TaskSpec) -> str:
    title = f"[{spec.task_type}] {spec.objective[:65]}"
    lines = [
        f"## {title}",
        "",
        f"**Priority:** {spec.priority}",
        "",
        "### Objective",
        spec.objective,
        "",
        "### Must have",
    ]
    lines += [f"- [ ] `{x}`" for x in spec.must_have] or ["- _none extracted_"]
    lines += ["", "### Nice to have"]
    lines += [f"- {x}" for x in spec.nice_to_have] or ["- _none_"]
    lines += ["", "### Exclusions / guardrails"]
    lines += [f"- {x}" for x in spec.exclusions] or ["- _none_"]
    if spec.constraints:
        lines += ["", "### Constraints"]
        lines.extend(f"- {c}" for c in spec.constraints)
    if spec.output_format:
        lines += ["", f"### Output format\n`{spec.output_format}`"]
    lines += ["", f"### Human handoff\n{'Yes' if spec.human_handoff else 'No'}"]
    if spec.open_questions:
        lines += ["", "### Open questions"]
        lines.extend(f"- {q}" for q in spec.open_questions)
    return "\n".join(lines)

In [10]:
def task_spec_to_jira_fields(spec: TaskSpec) -> dict[str, Any]:
    """Shape similar to Atlassian REST payloads (labels/summary/description only)."""
    summary = f"[{spec.task_type}] {spec.objective[:200]}"
    labels = sorted(
        {spec.task_type, spec.priority}
        | {f"must:{x}" for x in spec.must_have[:8]}
        | {f"exclude:{x}" for x in spec.exclusions[:8]}
    )
    description = task_spec_to_github_issue(spec)
    return {
        "fields": {
            "summary": summary,
            "description": {"type": "doc", "version": 1, "content": [{"type": "paragraph", "content": [{"type": "text", "text": description[:32000]}]}]},
            "labels": labels[:20],
        }
    }


def task_spec_to_jira_simple(spec: TaskSpec) -> dict[str, Any]:
    """Flat JSON for teaching / mock APIs (easier to read than ADF)."""
    return {
        "summary": f"[{spec.task_type}] {spec.objective[:200]}",
        "priority": spec.priority,
        "labels": [spec.task_type, spec.priority],
        "description_markdown": task_spec_to_github_issue(spec),
    }

In [11]:
def _yaml_escape(s: str) -> str:
    return s.replace("\\", "\\\\").replace('"', '\\"')


def task_spec_to_yaml_sketch(spec: TaskSpec) -> str:
    lines = [
        f"task_type: {spec.task_type}",
        f"priority: {spec.priority}",
        f'objective: "{_yaml_escape(spec.objective)}"',
        "must_have:",
        *[f"  - {x}" for x in spec.must_have],
        "nice_to_have:",
        *[f"  - {x}" for x in spec.nice_to_have],
        "exclusions:",
        *[f"  - {x}" for x in spec.exclusions],
        "constraints:",
        *[f"  - {x}" for x in spec.constraints],
        f"human_handoff: {str(spec.human_handoff).lower()}",
    ]
    if spec.output_format:
        lines.append(f'output_format: "{_yaml_escape(spec.output_format)}"')
    if spec.open_questions:
        lines.append("open_questions:")
        lines.extend(f"  - {_yaml_escape(q)}" for q in spec.open_questions)
    return "\n".join(lines)

In [12]:
def task_spec_to_agent_workflow_brief(spec: TaskSpec) -> str:
    """High-level bullets for LangGraph / workflow design discussions."""
    steps = [
        f"1. **Classify** incoming user intent against task `{spec.task_type}`.",
        "2. **Satisfy must_have** via tools or retrieval:",
    ]
    if spec.must_have:
        steps.extend(f"   - {m}" for m in spec.must_have)
    else:
        steps.append("   - _(define tools after open questions)_")
    if spec.exclusions:
        steps.append("3. **Guardrails**: refuse or escalate when topic matches:")
        steps.extend(f"   - {e}" for e in spec.exclusions)
    else:
        steps.append("3. **Guardrails**: none specified — add policy review.")
    if spec.human_handoff:
        steps.append("4. **HITL**: route to human queue when confidence low or exclusion hit.")
    else:
        steps.append("4. **HITL**: not required by spec (revisit for production).")
    if spec.open_questions:
        steps.append("5. **Open discovery** before build:")
        steps.extend(f"   - {q}" for q in spec.open_questions)
    return "\n".join(steps)

## Run: requests A / B / C

Execute to see **live** model output. Shapes below are **representative** of what a well-calibrated run looks like; your labels may differ slightly.

| Request | Expected `task_type` |
|--------|------------------------|
| A | `dashboard` |
| B | `chatbot` |
| C | `script` |

In [13]:
REQUESTS = {
    "A": (
        "Can we build a small dashboard for sales that shows trends by region and product line? "
        "It should be simple and export to PDF."
    ),
    "B": (
        "I need a chatbot for support docs, but it should not answer billing questions and "
        "it should hand those off to a person."
    ),
    "C": (
        "Make a script that cleans our customer CSVs because the country field is a mess and "
        "dates come in different formats."
    ),
}

extracted: dict[str, TaskSpec] = {}
for label, text in REQUESTS.items():
    spec = extract_task_spec(text)
    extracted[label] = spec
    issues = validate_task_spec(spec)
    print(f"======== Request {label} ========")
    print(spec.model_dump_json(indent=2))
    if issues:
        print("validation notes:", issues)
    print()

======== Request A ========
{
  "task_type": "dashboard",
  "objective": "Create a small sales dashboard that visualizes trends by region and product line.",
  "priority": "medium",
  "must_have": [
    "show_trends",
    "filter_by_region",
    "filter_by_product_line",
    "simple_design",
    "pdf_export"
  ],
  "nice_to_have": [],
  "exclusions": [],
  "constraints": [
    "simple_design"
  ],
  "output_format": "pdf_export",
  "human_handoff": false,
  "open_questions": [
    "What data source(s) should the dashboard connect to?",
    "Which specific sales metrics (e.g., revenue, units sold) need to be displayed?",
    "What time range or granularity (daily, weekly, monthly) is required for the trends?",
    "Should the dashboard be interactive (e.g., drill‑down, hover tooltips) or static?",
    "Are there any branding or style guidelines to follow?",
    "Who are the intended users and what access controls are needed?"
  ]
}

======== Request B ========
{
  "task_type": "chatbot"

## Downstream previews (same spec, four artifacts)

In [14]:
demo = extracted.get("B") or extract_task_spec(REQUESTS["B"])

print("=== GitHub issue markdown ===\n")
print(task_spec_to_github_issue(demo))
print("\n=== Jira (flat teaching JSON) ===\n")
print(json.dumps(task_spec_to_jira_simple(demo), indent=2))
print("\n=== YAML sketch ===\n")
print(task_spec_to_yaml_sketch(demo))
print("\n=== Agent workflow brief ===\n")
print(task_spec_to_agent_workflow_brief(demo))

=== GitHub issue markdown ===

## [chatbot] Provide a chatbot that answers support‑documentation questions wh

**Priority:** medium

### Objective
Provide a chatbot that answers support‑documentation questions while routing any billing inquiries to a human.

### Must have
- [ ] `support_doc_knowledge`
- [ ] `human_hand_off`

### Nice to have
- _none_

### Exclusions / guardrails
- billing_questions

### Human handoff
Yes

### Open questions
- Which deployment channels (web, mobile app, Slack, Teams, etc.) are required?
- What is the source and format of the support documentation (e.g., Confluence, markdown, PDF)?
- Should the chatbot support multiple languages or only English?
- Are there any performance or latency SLAs to meet?
- What branding, tone, or personality guidelines should the bot follow?
- Is user authentication or context needed for answering certain queries?
- How exactly should the handoff to a human be performed (ticket creation, live‑chat transfer, email notification, 

## Stretch implementation: vocabulary hints per `task_type`

Tightening list tokens reduces drift when you integrate with fixed enums in Jira or internal routers.

In [15]:
TASK_VOCAB: dict[str, dict[str, list[str]]] = {
    "dashboard": {
        "must_have_suggestions": [
            "trends_by_region",
            "trends_by_product_line",
            "pdf_export",
            "sales_metrics",
        ],
    },
    "chatbot": {
        "must_have_suggestions": ["support_docs_grounding", "human_escalation_path"],
        "exclusion_suggestions": ["billing_questions"],
    },
    "script": {
        "must_have_suggestions": [
            "normalize_country_field",
            "normalize_date_formats",
            "batch_csv_processing",
        ],
    },
}


def build_system_prompt_with_vocab() -> str:
    base = build_system_prompt()
    vocab_block = json.dumps(TASK_VOCAB, indent=2)
    return (
        base
        + "\n\nOptional vocabulary hints (prefer these tokens when they fit; still obey user text):\n"
        + vocab_block
    )


def extract_task_spec_vocab(message: str) -> TaskSpec:
    chain = llm.with_structured_output(TaskSpec)
    return chain.invoke(
        [SystemMessage(content=build_system_prompt_with_vocab()), HumanMessage(content=message)]
    )


# Compare A with and without vocab (optional)
print("--- Without extra vocab ---")
print(extract_task_spec(REQUESTS["A"]).must_have)
print("--- With TASK_VOCAB ---")
print(extract_task_spec_vocab(REQUESTS["A"]).must_have)

--- Without extra vocab ---
['show_trends', 'filter_by_region', 'filter_by_product_line', 'simple_design', 'export_to_pdf']
--- With TASK_VOCAB ---
['trends_by_region', 'trends_by_product_line', 'pdf_export']


## Appendix: when `task_type` is too narrow

Bug / research / multi-capability messages often need either:

- **`task_type: "other"`** (extend the `Literal`), or
- a **second model** for bug vs feature vs research, or
- heavy use of **`open_questions`** and a human triage queue.

Try the exercise appendix strings with this notebook’s schema: you should see **honest** `open_questions` rather than a forced wrong `task_type`—if the model still forces a bad bucket, tighten the prompt (“use open_questions if none of dashboard/chatbot/script fits”).